# **Limpeza e Extração de Dados das Súmulas CBF**

## **1. Configuração do Ambiente**

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import pdfplumber

## **2. Configuração**

Carrega o índice de súmulas gerado no notebook 0. Cada linha do índice corresponde a uma súmula baixada, com metadados de jogo e caminho do PDF.

In [2]:
DATA_PATH = Path("../data")
RAW_PATH = DATA_PATH / "raw" / "match-reports"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

INDEX_PATH = RAW_PATH / "match_reports_index.csv"
OUTPUT_PATH = PROCESSED_PATH / "sumulas_clean.csv"

index = pd.read_csv(INDEX_PATH)
index = index[index["download_status"] == "exists"].reset_index(drop=True)

print(f"Súmulas disponíveis: {len(index)}")
index.head(2)

Súmulas disponíveis: 2427


,year,competition_id,round,game_id,game_number,home_team,away_team,match_date,match_time,venue,document_title,document_title_normalized,document_url,file_path,download_status
0,2020,12464,1,815072,1,Flamengo,Atlético Mineiro,09/08/2020,16:00,Maracanã - Rio de Janeiro - RJ,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/1421s...,../data/raw/match-reports/2020/round-01/2020_r...,exists
1,2020,12464,1,815073,2,Botafogo de Futebol E Regatas,Esporte Clube Bahia,30/09/2020,21:30,Nilton Santos - Rio de Janeiro - RJ,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/1422s...,../data/raw/match-reports/2020/round-01/2020_r...,exists


## **3. Extração de Texto dos PDFs**

Cada súmula é um PDF de 2–3 páginas. O texto é extraído com `pdfplumber` e as páginas são concatenadas em uma única string para simplificar o parsing com regex.

In [3]:
def extract_pdf_text(file_path: str) -> str:
    try:
        with pdfplumber.open(file_path) as pdf:
            return "\n".join(page.extract_text() or "" for page in pdf.pages)
    except Exception:
        return ""

## **4. Funções de Parsing**

### 4.1 Utilitários de Normalização

O matching de nomes de times usa normalização (sem acento, minúsculo) para lidar com variações entre o índice e o texto do PDF — por exemplo, `"Fortaleza Esporte Clube"` no índice vs. `"Fortaleza/CE"` no PDF.

In [4]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text.casefold().strip()


def _prep_for_match(text: str) -> str:
    n = normalize_text(text)
    n = re.sub(r"/[a-z]{2,3}\s*$", "", n).rstrip()
    return re.sub(r"[^\w\s]", "", n).strip()


def identify_team(line: str, home_team: str, away_team: str) -> str | None:
    line_clean = _prep_for_match(line)
    home_n = re.sub(r"[^\w\s]", "", normalize_text(home_team)).strip()
    away_n = re.sub(r"[^\w\s]", "", normalize_text(away_team)).strip()

    def score(team_n: str, line_c: str) -> float:
        if line_c.endswith(team_n):
            return 3.0
        tokens = [t for t in team_n.split() if len(t) > 3]
        last3 = " ".join(line_c.split()[-3:])
        for tok in tokens:
            if line_c.endswith(tok):
                return 1.0
        for tok in tokens:
            if tok in last3:
                return 0.5
        return 0.0

    hs = score(home_n, line_clean)
    as_ = score(away_n, line_clean)
    if hs > as_:
        return home_team
    if as_ > hs:
        return away_team
    return None

### 4.2 Campos Simples

Extrai árbitro e data diretamente do texto do PDF via regex.

In [5]:
def parse_arbitro(text: str) -> str | None:
    m = re.search(r"Arbitro:\s*([^(\n]+)", text)
    return m.group(1).strip() if m else None


def parse_match_date(match_date_raw: str) -> str:
    try:
        d, mo, y = str(match_date_raw).strip().split("/")
        return f"{y}-{mo}-{d}"
    except Exception:
        return match_date_raw

### 4.3 Parsing de Cartões

Cada entrada de cartão no PDF tem um cabeçalho com timestamp e time, seguido de um campo `Motivo:` potencialmente multi-linha. O prefixo de código (`A1.4.`, `V1.`, etc.) é removido do motivo.

- **Amarelos**: seção entre `Cartões Amarelos` e `Cartões Vermelhos`
- **Vermelhos**: seção entre `Cartões Vermelhos` e `Ocorrências / Observações`

In [6]:
TIMESTAMP_RE = re.compile(r"^\d{2}:\d{2}\s+[12]T\s+")
SKIP_RE = re.compile(r"^(Tempo 1T|NÃO HOUVE|[A-Z]{2,3} -|Confederação|\d+\s*/\s*\d+)")


def clean_motivo(raw: str) -> str:
    cleaned = re.sub(r"^[A-Z]\d*(?:\.\d+)*\.?\s*", "", raw.strip())
    return cleaned.strip()


def get_section(text: str, start_marker: str, end_marker: str) -> str:
    m = re.search(
        rf"{re.escape(start_marker)}\n(.+?)(?={re.escape(end_marker)})",
        text,
        re.DOTALL,
    )
    return m.group(1) if m else ""


def parse_card_section(
    section_text: str, home_team: str, away_team: str
) -> list[tuple[str, str]]:
    cards = []
    current_team = None
    motivo_lines: list[str] = []
    in_motivo = False

    for line in section_text.split("\n"):
        line = line.strip()
        if not line or SKIP_RE.match(line):
            continue

        if TIMESTAMP_RE.match(line):
            if current_team and motivo_lines:
                cards.append((current_team, clean_motivo(" ".join(motivo_lines))))
            current_team = identify_team(line, home_team, away_team)
            motivo_lines = []
            in_motivo = False

        elif "Motivo:" in line:
            in_motivo = True
            raw = re.sub(r"^.*?Motivo:\s*", "", line)
            motivo_lines = [raw] if raw else []

        elif in_motivo and not SKIP_RE.match(line):
            motivo_lines.append(line)

    if current_team and motivo_lines:
        cards.append((current_team, clean_motivo(" ".join(motivo_lines))))

    return cards

## **5. Construção do Dataset**

Para cada súmula, são geradas **duas linhas** — uma para cada time como `focal_team`. Os motivos de cartão são ancorados ao time focal: `yellow_reason_1..N` e `red_reason_1..M` são os motivos dos cartões recebidos pelo time focal naquele jogo.

In [7]:
def process_match(row: pd.Series, text: str) -> list[dict]:
    home_team = row["home_team"]
    away_team = row["away_team"]

    yellow_section = get_section(text, "Cartões Amarelos", "Cartões Vermelhos")
    red_section = get_section(text, "Cartões Vermelhos", "Ocorrências / Observações")

    yellow_cards = parse_card_section(yellow_section, home_team, away_team)
    red_cards = parse_card_section(red_section, home_team, away_team)

    arbitro = parse_arbitro(text)
    match_date = parse_match_date(row["match_date"])

    records = []
    for focal_team in [home_team, away_team]:
        yellow_reasons = [m for t, m in yellow_cards if t == focal_team]
        red_reasons = [m for t, m in red_cards if t == focal_team]

        record: dict = {
            "year": row["year"],
            "match_date": match_date,
            "home_team": home_team,
            "away_team": away_team,
            "focal_team": focal_team,
            "arbitro": arbitro,
        }
        for i, reason in enumerate(yellow_reasons, 1):
            record[f"yellow_reason_{i}"] = reason
        for i, reason in enumerate(red_reasons, 1):
            record[f"red_reason_{i}"] = reason

        records.append(record)

    return records

In [8]:
all_records = []
failed = []

for position, (_, row) in enumerate(index.iterrows(), start=1):
    text = extract_pdf_text(row["file_path"])
    if not text:
        failed.append(row["file_path"])
        continue
    all_records.extend(process_match(row, text))
    if position % 200 == 0:
        print(f"Progresso: {position}/{len(index)}")

print(f"\nProcessadas: {len(index) - len(failed)} súmulas")
print(f"Falhas na leitura: {len(failed)}")
print(f"Registros gerados: {len(all_records)}")

Progresso: 200/2427
Progresso: 400/2427
Progresso: 600/2427
Progresso: 800/2427
Progresso: 1000/2427
Progresso: 1200/2427
Progresso: 1400/2427
Progresso: 1600/2427
Progresso: 1800/2427
Progresso: 2000/2427
Progresso: 2200/2427
Progresso: 2400/2427

Processadas: 2427 súmulas
Falhas na leitura: 0
Registros gerados: 4854


In [9]:
dataset = pd.DataFrame(all_records)

yellow_cols = sorted(
    [c for c in dataset.columns if c.startswith("yellow_reason_")],
    key=lambda c: int(c.split("_")[-1]),
)
red_cols = sorted(
    [c for c in dataset.columns if c.startswith("red_reason_")],
    key=lambda c: int(c.split("_")[-1]),
)

ordered_cols = ["year", "match_date", "home_team", "away_team", "focal_team", "arbitro"] + yellow_cols + red_cols
dataset = dataset[ordered_cols]

print(f"Shape: {dataset.shape}")
print(f"Colunas de amarelo: {len(yellow_cols)} ({yellow_cols[0]} … {yellow_cols[-1]})")
print(f"Colunas de vermelho: {len(red_cols)} ({red_cols[0]} … {red_cols[-1]})")
dataset.head(4)

Shape: (4854, 17)
Colunas de amarelo: 8 (yellow_reason_1 … yellow_reason_8)
Colunas de vermelho: 3 (red_reason_1 … red_reason_3)


,year,match_date,home_team,away_team,focal_team,arbitro,yellow_reason_1,yellow_reason_2,yellow_reason_3,yellow_reason_4,yellow_reason_5,yellow_reason_6,yellow_reason_7,yellow_reason_8,red_reason_1,red_reason_2,red_reason_3
0,2020,2020-08-09,Flamengo,Atlético Mineiro,Flamengo,Raphael Claus,Dar ou tentar dar uma rasteira ou um calço em ...,Dar ou tentar dar uma rasteira ou um calço em ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020,2020-08-09,Flamengo,Atlético Mineiro,Atlético Mineiro,Raphael Claus,Reclamar / protestar (verbalmente ou por gesto...,"Outro motivo (somente neste caso, abriria um c...",Cometer uma falta tática para impedir um ataqu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020,2020-09-30,Botafogo de Futebol E Regatas,Esporte Clube Bahia,Botafogo de Futebol E Regatas,Savio Pereira Sampaio,Dar ou tentar dar uma rasteira ou um calço em ...,Dar ou tentar dar uma rasteira ou um calço em ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020,2020-09-30,Botafogo de Futebol E Regatas,Esporte Clube Bahia,Esporte Clube Bahia,Savio Pereira Sampaio,Dar ou tentar dar uma rasteira ou um calço em ...,Dar ou tentar dar uma rasteira ou um calço em ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## **6. Validação**

Verificação rápida de cobertura: quantos registros têm árbitro preenchido, quantos têm pelo menos um cartão amarelo/vermelho, e distribuição de cartões por ano.

In [10]:
has_arbitro = dataset["arbitro"].notna().sum()
has_yellow = dataset[yellow_cols].notna().any(axis=1).sum()
has_red = dataset[red_cols].notna().any(axis=1).sum()

print(f"Total de registros (focal_team rows): {len(dataset)}")
print(f"Com árbitro preenchido: {has_arbitro} ({has_arbitro/len(dataset):.1%})")
print(f"Com ao menos 1 amarelo: {has_yellow} ({has_yellow/len(dataset):.1%})")
print(f"Com ao menos 1 vermelho: {has_red} ({has_red/len(dataset):.1%})")

Total de registros (focal_team rows): 4854
Com árbitro preenchido: 4854 (100.0%)
Com ao menos 1 amarelo: 4382 (90.3%)
Com ao menos 1 vermelho: 458 (9.4%)


In [11]:
yellow_count = dataset[yellow_cols].notna().sum(axis=1)
red_count = dataset[red_cols].notna().sum(axis=1)

stats = (
    dataset.assign(n_yellow=yellow_count, n_red=red_count)
    .groupby("year")[["n_yellow", "n_red"]]
    .mean()
    .round(2)
)
stats.columns = ["média amarelos/time", "média vermelhos/time"]
stats

,média amarelos/time,média vermelhos/time
year,,
2020,1.98,0.11
2021,2.06,0.09
2022,2.24,0.12
2023,2.28,0.10
2024,2.27,0.11
2025,2.15,0.10
2026,2.02,0.12


## **7. Salvamento**

Dataset final salvo em `data/processed/sumulas_clean.csv`.

In [12]:
dataset.to_csv(OUTPUT_PATH, index=False)
print(f"Salvo em: {OUTPUT_PATH}")
print(f"Shape final: {dataset.shape}")

Salvo em: ../data/processed/sumulas_clean.csv
Shape final: (4854, 17)
